# Phase 8.4 — Apriori: Association Rules for Crime Patterns

**Goal:** find combinations of conditions (place, time, season, district) that tend to go together with a crime type, for example *commercial place + afternoon → THEFT*.

**Input:** `data/processed/categorical.parquet` — 150,000 sampled incidents with a known time, one-hot encoded.
**Script:** `ml/train_apriori.py` (this notebook reads its saved results; it does not rerun the mining).

## 1. Method and measures

Apriori first finds *frequent itemsets* (combinations that occur in enough incidents), then builds rules `A → B` from them.

- **Support:** share of all incidents that contain both A and B.
- **Confidence:** among incidents with A, the share that also contain B.
- **Lift:** confidence divided by the overall share of B. Lift above 1 means A and B occur together more often than chance; the larger, the stronger the association.

**Settings:** `min_support = 0.01` (inside the plan's 0.01–0.02 range), `min_confidence = 0.30`, itemsets of at most 4 items, rules ranked by lift, only lift > 1 kept.

**Why confidence is 0.30 and not the planned 0.50:** at 0.50 only 4 conditions → crime-type rules existed, all saying "commercial place → THEFT". No single crime group is common enough to reach 50% confidence under most conditions (THEFT, the largest, is only about 22% of incidents). Lowering to 0.30 gives 34 rules covering more than one crime group, while lift (above 1) remains the ranking measure.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BASE = ".." if os.path.basename(os.getcwd()) == "notebooks" else "."
RESULTS, MODELS = f"{BASE}/ml/results", f"{BASE}/ml/models"
pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 80)

## 2. Input data

In [ ]:
df = pd.read_parquet(f"{BASE}/data/processed/categorical.parquet").astype(bool)
print("Incidents:", len(df), "| one-hot columns:", df.shape[1])
groups = pd.Series([c.split("=")[0] for c in df.columns]).value_counts()
print(groups.to_string())
print("\nMost common crime groups in the sample (share of incidents):")
print((df[[c for c in df.columns if c.startswith("crime_group=")]].mean() * 100).round(1).sort_values(ascending=False).to_string())

## 3. Rules found

In [ ]:
allr = pd.read_csv(f"{RESULTS}/apriori_all_rules.csv")
allr["consequent_type"] = allr["consequent_str"].str.split("=").str[0]
print("All rules (lift > 1):", len(allr))
print(allr["consequent_type"].value_counts().to_string())
crime = pd.read_csv(f"{MODELS}/rules.csv")
print("\nCrime-pattern rules (conditions -> crime group), saved for the API:", len(crime))
print(crime["consequent_str"].value_counts().to_string())

### Top 10 crime-pattern rules by lift

In [ ]:
top10 = pd.read_csv(f"{RESULTS}/apriori_top10_lift.csv")
display(top10[["antecedent_str", "consequent_str", "support", "confidence", "lift"]].round(3))

fig, ax = plt.subplots(figsize=(10, 5))
labels = (top10["antecedent_str"].str.replace("location_group=", "").str.replace("time_of_day=", "")
          .str.replace("is_weekend=", "").str.replace("season=", "").str.replace("district=", "")
          + "  →  " + top10["consequent_str"].str.replace("crime_group=", ""))
ax.barh(labels[::-1], top10["lift"][::-1])
ax.axvline(1, color="gray", linestyle="--"); ax.set_xlabel("Lift (1 = no association)")
ax.set_title("Top 10 crime-pattern rules by lift")
plt.tight_layout(); plt.show()

## 4. Check one rule by hand

In [ ]:
r = crime.sort_values("lift", ascending=False).iloc[0]
ante = r["antecedent_str"].split(" AND ")
cons = r["consequent_str"]
a = df[ante].all(axis=1); c = df[cons]
support = (a & c).mean(); confidence = (a & c).sum() / a.sum(); lift = confidence / c.mean()
print("Rule:", r["antecedent_str"], "->", cons)
print(f"  support    saved {r['support']:.4f}  recomputed {support:.4f}")
print(f"  confidence saved {r['confidence']:.4f}  recomputed {confidence:.4f}")
print(f"  lift       saved {r['lift']:.4f}  recomputed {lift:.4f}")
print(f"  overall share of {cons}: {c.mean()*100:.1f}%  |  share among matching incidents: {confidence*100:.1f}%")

## 5. Interpretation and limitations

- **Two stories appear in the top rules.** (1) **Commercial places go with THEFT:** commercial + afternoon (support 3.4%, confidence 56%, lift 2.5), also with District 18 (confidence 60%, lift 2.7) and District 1 (56%, lift 2.5), with variants for morning, summer and fall. THEFT is about 22% of all incidents, so these conditions raise its share to roughly half. (2) **District 11 + street locations go with "ALL OTHER"** (lift 3.1, confidence 45%, support 1.3%). "ALL OTHER" is a catch-all category, so this rule says more about District 11 than about a type of crime.
- **Many top rules are variants of one pattern** (commercial → THEFT with a different time or season attached). They are kept because each adds a condition; they are not independent findings.
- **Lift is moderate (2 to 3),** and support is low for the district rules (about 1%), so these patterns apply to a small share of incidents.
- **Descriptive, not causal:** a rule shows that conditions and a crime type occur together in recorded incidents. It does not show that the place or time causes the crime.
- **Sample and scope:** 150,000 incidents with a known time (a sample, not the full data); itemsets limited to 4 items; threshold choices (support 0.01, confidence 0.30) affect which rules appear.
- **Use on the website:** the Patterns page shows the top rules by lift from `ml/models/rules.csv` with this wording.